# Top People EDA

## Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import networkx as nx
from networkx.algorithms import community

from collections import Counter
from itertools import combinations
from IPython.display import display  

from fuzzywuzzy import fuzz
from itertools import combinations

In [3]:
df = pd.read_parquet("../data_storage/final_data/final_dataset_with_attribution.parquet")

In [4]:
people = pd.read_csv("../data_storage/final_data/persons_detected.csv")

In [5]:
people_row = pd.read_csv("../data_storage/final_data/persons_by_row.csv")

## People Mentioned

In [6]:
people.head()

,person,count
0,trump,31486
1,kennedy,20997
2,donald trump,10157
3,robert kennedy,9382
4,biden,4422


In [7]:
pd.unique(people.values.ravel())

array(['trump', 31486, 'kennedy', ..., 'royal cannon', 'ekbo',
       'dahw medium'], dtype=object)

## Top People

In [8]:
# Ensure types match
people_row["row_index"] = people_row["row_index"].astype("int64")
df["seq_index"]        = df["seq_index"].astype("int64")

# If persons is a comma-separated string, normalize to list
if people_row["persons"].dtype == "string" or people_row["persons"].dtype == object:
    people_row["persons"] = people_row["persons"].fillna("").apply(
        lambda s: [p.strip() for p in str(s).split(",") if p.strip()]
    )

merged = people_row.merge(
    df,
    left_on="row_index",
    right_on="seq_index",
    how="left"
)

In [9]:
merged.head(5)

,row_index,tag_name_x,persons,has_person_x,article_body,article_id,author_name,channel_name,circulation_size,feed_name,...,vipr_score,vipr_weight,headline_token_count,body_token_count,token_count,seq_index,has_person_y,is_conversion,circulation_size_bin,sentiment_score_bin
0,0,public_health,"[david miscavige, david miscavige, castle kyal...",1,johannesburg south africa los angeles calif ja...,17875604184,unknown,web,800,opoint,...,23400,900,13,510,523,0.0,True,True,CIRCULATION_SIZE_Q1,SENTIMENT_SCORE_Q5
1,0,public_health,"[david miscavige, david miscavige, castle kyal...",1,bhopal madhya pradesh february madhya pradesh ...,17996175185,unknown,web,1173,opoint,...,-3000,1500,12,215,227,0.0,True,True,CIRCULATION_SIZE_Q2,SENTIMENT_SCORE_Q3
2,0,public_health,"[david miscavige, david miscavige, castle kyal...",1,abuja nigeria nigeria received million vaccine...,19281265101,unknown,web,2203,opoint,...,-6000,1200,7,213,220,0.0,True,True,CIRCULATION_SIZE_Q3,SENTIMENT_SCORE_Q3
3,0,public_health,"[david miscavige, david miscavige, castle kyal...",1,item include chrysler town country newport cub...,19620614081,unknown,broadcast,320797,tveyes,...,3024,216,2,384,386,0.0,True,True,CIRCULATION_SIZE_Q5,SENTIMENT_SCORE_Q5
4,0,public_health,"[david miscavige, david miscavige, castle kyal...",1,article taking look healthiest country world w...,18000221773,unknown,web,14562,opoint,...,5500,550,3,1723,1726,0.0,True,True,CIRCULATION_SIZE_Q3,SENTIMENT_SCORE_Q4


In [12]:
person_sentiment = (
    merged.explode("persons")
    .groupby("persons", as_index=False)
    .agg(avg_sent=("sentiment_score", "mean"),
         pos_rate=("sentiment_band", lambda x: (x=="positive").mean()),
         mentions=("article_id", "count"))
    .sort_values("mentions", ascending=False)
)

In [13]:
person_sentiment.head()

,persons,avg_sent,pos_rate,mentions
150812,trump,-7.628568,0.171424,27709
82003,kennedy,-6.307978,0.185288,18339
40505,donald trump,-7.835701,0.166201,8941
127082,robert kennedy,-7.270346,0.173317,8245
15565,biden,-7.994137,0.172317,3923
